# 00 — NLP Learning Map: Raw Language to Production

**Learning objective.** Build the mental model for the complete NLP lifecycle before learning individual techniques.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**business question → task/data contract → representation/model → evaluation → inference**

The key question is not “which API do I call?” but **which representation changes next when I change a control?**

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change the **business question** | the target, labels and acceptable errors change | the entire downstream pipeline may need to change |
| Change the **evaluation metric** | which errors are rewarded/penalized changes | model selection can reverse even with the same predictions |
| Change the **split strategy** | what 'unseen' means changes | random, group or temporal generalization answer different questions |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. If a model has high accuracy but terrible minority-class recall, would the system still be acceptable?
2. If user IDs repeat across train/test, what kind of generalization are you actually measuring?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Use the lifecycle map whenever you are turning a language problem into an ML system.

### When not to use / caution
Do not start with a favorite model before the target, split and metric are defined.

### Debugging lens
When a downstream result looks surprising, walk backward: metric → predictions → representation → preprocessing → split → labels → business question.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


## The end-to-end NLP lifecycle

```text
business question
     ↓
raw language + labels/metadata
     ↓
data quality / leakage / split strategy
     ↓
cleaning + normalization
     ↓
tokenization
     ↓
linguistic / statistical / neural representation
     ↓
model
     ↓
task-specific decoding
     ↓
evaluation + error analysis
     ↓
inference contract
     ↓
monitoring, drift, feedback, retraining
```

NLP is not synonymous with “use a transformer.” Different tasks can be solved with rules, sparse vectors, classical ML, embeddings, sequence models, or transformers. The correct abstraction is a **language system with an explicit data and evaluation contract**.

In [2]:
stages = pd.DataFrame([
 ('Text quality','Unicode, casing, URLs, punctuation','01–04'),
 ('Structure','tokens, sentences, n-grams, syntax, entities','03–08'),
 ('Representation','BoW, TF-IDF, dense embeddings','09–13'),
 ('Modeling','classical ML, RNN/LSTM/GRU, attention, transformers','14–20'),
 ('Applications','sentiment, topics, search, IE, QA, summarization','15–22'),
 ('Assurance','metrics, errors, leakage, calibration','23'),
 ('Operations','serialization, inference, drift, monitoring, robustness','24–29'),
], columns=['layer','questions','notebooks'])
stages

            layer  ... notebooks
0    Text quality  ...     01–04
1       Structure  ...     03–08
2  Representation  ...     09–13
3        Modeling  ...     14–20
4    Applications  ...     15–22
5       Assurance  ...        23
6      Operations  ...     24–29

[7 rows x 3 columns]

In [3]:
print('Track notebooks:', 30)
print('Contract: every notebook is executable and stores its outputs for GitHub rendering.')

Track notebooks: 30
Contract: every notebook is executable and stores its outputs for GitHub rendering.


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Place any NLP technique inside the wider lifecycle
- Choose a baseline before escalating model complexity